# Controlled 200M query-path evidence audit
## tl;dr
Five fresh processes passed all 164 million timed-operation checks. Median paired
binary/learned ratios are 1.287 for uniform membership, 1.278 for exact rank and
1.329 for range count. These are conditional same-state query-path comparisons,
not specialist or end-to-end index comparisons. Three processes have sampled
Windows page-out activity; all five are retained.
## Context & Methods
The experimental unit is a fresh process. Four equal-size batches per path are
averaged before summarizing five processes. A paired percentile bootstrap uses
10,000 process resamples, seed 20260905. With n=5 these descriptive intervals are
coarse and do not capture dataset, hardware, seed or host-paging uncertainty.
### Key Assumptions
Input files, source hashes and query streams are fixed. No cache flush is claimed.
Host counters describe the whole Windows machine; positive page-out is a resource
warning, not proof that a particular timed query was swapped. The first launch
failed before executing the benchmark and is documented separately.
## Data
Raw records and telemetry are retained under results_q1/controlled_queries_20260905
and results_q1/reassessment_20260905. The next cell displays and executes the exact
validation and aggregation code; it does not rerun a benchmark or start WSL.


In [1]:
import sys, inspect, json
from pathlib import Path
import pandas as pd
root = Path.cwd()
assert (root / 'scripts/analyze_controlled_queries.py').is_file()
sys.path.insert(0, str(root / 'scripts'))
from analyze_controlled_queries import analyze
print(inspect.getsource(analyze))
audit = analyze()
saved = json.loads((root / 'results_q1/controlled_queries_20260905/analysis.json').read_text())
assert audit == saved
print('Raw records, checksums, source identity, guest swap and saved aggregate agree.')

def analyze():
    records = []
    profiles = []
    hashes = {}
    for trial in range(2, 7):
        path = DATA / f"trial_{trial}.json"
        hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
        record = json.loads(path.read_text())
        result = record["result"]
        assert record["exit_code"] == 0 and result["all_oracles_passed"]
        assert result["selected_keys"] == 200000000
        assert result["base_keys"] == 199900000 and result["epsilon"] == 64
        assert result["fingerprint_capacity"] == result["final_mutation_count"] == 0
        assert result["query_paths"] == "learned_binary"
        assert result["queries"] == 1000000 and result["range_queries"] == 100000
        assert result["seed"] == 20260905 and result["model_segments"] == 3008305
        assert len(result["rows"]) == 8
        assert {(row["trial"], row["mode"]) for row in result["rows"]} == {
            (i, mode) for i in range(1, 5) for mode in ("learned", "binary")}
     

## Results
### Process-level resource checks

In [2]:
pd.DataFrame(audit['profiles'])

,trial,host_samples,host_min_available_mib,host_pageout_samples,host_max_pageout_per_second,guest_swap_counter_change,max_rss_kib,user_seconds,system_seconds,major_faults,minor_faults,voluntary_context_switches,involuntary_context_switches
0,2,48,313,2,11950,False,6280944,211.300104,7.317980,72418,580752,72714,471
1,3,45,529,1,4857,False,6280940,201.602225,6.032583,72418,580751,72712,455
2,4,46,1360,0,0,False,6280940,199.935467,5.932423,72397,580764,72694,742
3,5,47,1352,0,0,False,6280964,204.659453,6.120357,72397,580762,72694,745
4,6,46,1214,1,18144,False,6280964,202.119352,6.348345,72397,580762,72696,642


### Paired query-path comparison (latency in microseconds)

In [3]:
pd.DataFrame([{'metric': name,
    'learned_median_us': value['learned']['median_ns']/1000,
    'binary_median_us': value['binary']['median_ns']/1000,
    'median_paired_ratio': value['median_paired_speedup'],
    'paired_process_95_interval': value['paired_process_bootstrap_95']}
    for name, value in audit['metrics'].items()])

,metric,learned_median_us,binary_median_us,median_paired_ratio,paired_process_95_interval
0,uniform_membership_ns,5.981284,7.746001,1.287450,"[1.2384524338846945, 1.2986744124194016]"
1,balanced_membership_ns,3.703343,4.614349,1.246311,"[1.2122793916920487, 1.2621422642999882]"
2,deleted_membership_ns,0.966149,0.930381,0.962979,"[0.9546783525971071, 1.0012856800960122]"
3,uniform_rank_ns,6.656312,8.468915,1.278343,"[1.2669911875815936, 1.2966258858408906]"
4,range_ns,8.436419,11.203454,1.329363,"[1.304001686506468, 1.3673480455881943]"


### Mutation and construction phase times (seconds)

In [4]:
pd.DataFrame(audit['phases']).T

,seconds,median_seconds,min_seconds,max_seconds
construction_ms,"[62.047423588, 54.905140846, 55.50728325500000...",56.409688,54.905141,62.047424
insert_ms,"[0.816984057, 0.833333423, 0.801356794, 0.8426...",0.833333,0.801357,0.902578
base_delete_ms,"[0.917449607, 0.912852937, 0.875359417, 0.9542...",0.91745,0.875359,0.976154
base_reinsert_ms,"[0.888410987, 0.9232219629999999, 0.877490004,...",0.888411,0.866089,0.923222
inserted_delete_ms,"[0.139070157, 0.136651953, 0.17314827700000002...",0.142157,0.136652,0.173148


## Takeaways
- High confidence: five complete records share the frozen implementation and pass
  the full driver gates. No guest swap-counter change occurred.
- High analytical risk: Windows page-out was sampled in trials 2, 3 and 6. Minimum
  available memory was 313 MiB. No process is removed after seeing its timing.
- Learned recovery has lower process-average latency for the four non-tombstone
  streams in all five processes. Deleted-key membership checks the shared ledger
  first and has no consistent learned-path advantage.
- This repairs the missing learned-versus-binary measurement. It does not supply
  a specialist learned-string baseline, independent memory comparison, natural
  mixed-write sweep, tail latency, or a replicated consolidation measurement.
- Trial 1 is a failed prelaunch, not a missing or discarded performance result.
